# Import the Modules and Load the Cleaned data

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv('train_cleaned_phase1.csv')
test = pd.read_csv('test_cleaned_phase1.csv')

In [3]:
print(test.isnull().sum().sort_values(ascending=False))
train.isnull().sum().sort_values(ascending=False)

irradiance            615
soiling_ratio         610
maintenance_count     609
panel_age             607
current               587
temperature           582
cloud_coverage        582
module_temperature    580
voltage               547
humidity               83
pressure               79
wind_speed             69
id                      0
string_id               0
error_code              0
installation_type       0
dtype: int64


panel_age             987
maintenance_count     985
cloud_coverage        981
temperature           974
voltage               973
soiling_ratio         972
irradiance            967
module_temperature    943
current               941
pressure              131
humidity              122
wind_speed            117
id                      0
string_id               0
error_code              0
installation_type       0
efficiency              0
dtype: int64

In [4]:
zero_counts = (train == 0).sum().sort_values(ascending=False)
print("Number of zero values in each column:")
print(zero_counts)

Number of zero values in each column:
error_code            5792
voltage               4989
installation_type     4866
string_id             4755
temperature            374
maintenance_count      343
module_temperature      19
id                       1
irradiance               0
current                  0
soiling_ratio            0
humidity                 0
panel_age                0
pressure                 0
wind_speed               0
cloud_coverage           0
efficiency               0
dtype: int64


## null value treatment 
- We tried the KNNIputer as well but it gave us besst result for test File

In [6]:

# Data Cleaning
train.fillna({
    'temperature': train['temperature'].mean(),
    'irradiance': train['irradiance'].mean(),
    'panel_age': train['panel_age'].median(),
    'maintenance_count': train['maintenance_count'].median(),
    'soiling_ratio': train['soiling_ratio'].median(),
    'voltage': train['voltage'].mean(),
    'current': train['current'].mean(),
    'module_temperature': train['module_temperature'].mean(),
    'cloud_coverage': train['cloud_coverage'].median(),
    'error_code': train['error_code'].mode()[0],
    'installation_type': train['installation_type'].mode()[0]
}, inplace=True)

test.fillna({
    'temperature': test['temperature'].mean(),
    'irradiance': test['irradiance'].mean(),
    'panel_age': test['panel_age'].median(),
    'maintenance_count': test['maintenance_count'].median(),
    'soiling_ratio': test['soiling_ratio'].median(),
    'voltage': test['voltage'].mean(),
    'current': test['current'].mean(),
    'module_temperature': test['module_temperature'].mean(),
    'cloud_coverage': test['cloud_coverage'].median(),
    'error_code': test['error_code'].mode()[0],
    'installation_type': test['installation_type'].mode()[0]
}, inplace=True)


- Feature Engineering

In [7]:
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)



In [8]:
# Select only numeric columns (float or int)
numeric_cols = train.select_dtypes(include=['float64', 'float32', 'int64', 'int32'])

# Drop ID if it's just an identifier
numeric_cols = numeric_cols.drop(columns=['id'], errors='ignore')

# Compute correlation with 'efficiency'
eff_corr = numeric_cols.corr()['efficiency'].drop('efficiency').sort_values(key=abs, ascending=False)

# Show result
print("Correlation of features with efficiency:")
print(eff_corr)


Correlation of features with efficiency:
irradiance              0.757827
soiling_ratio           0.380091
current                 0.353273
power_output            0.306127
panel_age              -0.239400
voltage                 0.194301
irradiance_per_cloud    0.166737
humidity               -0.086020
module_temperature     -0.064242
temperature            -0.055789
string_id               0.020579
maintenance_count       0.016563
temp_diff              -0.012751
wind_speed             -0.009009
error_code              0.008187
installation_type      -0.005124
pressure               -0.004185
cloud_coverage         -0.003317
Name: efficiency, dtype: float64


## Train the Model

In [9]:

# Feature selection
drop_cols = ['id', 'efficiency']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [19]:
model = CatBoostRegressor(iterations=300, learning_rate=0.07, depth=5, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0450, Score: 95.50282200


- To find the best Prameter for the tuning 

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'iterations': [200, 300, 500]
}

cb = CatBoostRegressor(random_seed=42, verbose=False)

grid = GridSearchCV(cb, params, cv=3, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)

# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
# 2 Best params: {'depth': 6, 'iterations': 500, 'learning_rate': 0.03}

Best params: {'depth': 6, 'iterations': 500, 'learning_rate': 0.03}


In [17]:

# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=4, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0455, Score: 95.44908265


In [15]:
# 2. Best params: {'depth': 6, 'iterations': 500, 'learning_rate': 0.03}
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=6, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0449, Score: 95.50565206


In [20]:
from datetime import datetime
# Get current date and time
now = datetime.now()
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Final prediction
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'Test_submission_{timestamp}.csv', index=False)

submission_002437


# Case 2 with less Feature Selection 

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from datetime import datetime
import pandas as pd
import numpy as np

# Define feature sets
features_1 = ["irradiance", "soiling_ratio", "current", "power_output"]
features_2 = ["irradiance", "soiling_ratio", "current", "power_output","panel_age", "voltage", "irradiance_per_cloud", "temperature"] #-iska result best raha
features_3 = ["irradiance", "soiling_ratio", "current", "power_output","panel_age", "voltage", "irradiance_per_cloud","humidity", "module_temperature", "temperature"]

feature_sets = {
    "F1_Top": features_1,
    "F2_Moderate": features_2,
    "F3_Broader": features_3
}

# Define parameter sets
param_sets = {
    "P1_depth4": {"iterations": 500, "learning_rate": 0.03, "depth": 4},
    "P2_depth6": {"iterations": 500, "learning_rate": 0.05, "depth": 6},
    "P3_depth5": {"iterations": 300, "learning_rate": 0.07, "depth": 5}
}

# Loop through each feature set and param set
for feat_name, selected_features in feature_sets.items():
    for param_name, params in param_sets.items():
        combo_name = f"{feat_name}_{param_name}"
        print(f"\n🚀 Running {combo_name} with features: {selected_features} and params: {params}")

        # Prepare data
        X = train[selected_features]
        y = train['efficiency']
        X_test = test[selected_features]

        # Split train/val
        X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        # Initialize model
        model = CatBoostRegressor(**params, random_seed=42, verbose=False)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

        # Evaluate
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        score = (1 - rmse) * 100
        print(f"✅ RMSE: {rmse:.4f}, Score: {score:.8f}")

        # Predict on test and save submission
        preds = model.predict(X_test)
        timestamp = datetime.now().strftime("%H%M%S")
        submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})
        filename = f'Test_submission_{combo_name}_{timestamp}.csv'
        submission.to_csv(filename, index=False)
        print(f"📁 Submission saved: {filename}")





🚀 Running F1_Top_P1_depth4 with features: ['irradiance', 'soiling_ratio', 'current', 'power_output'] and params: {'iterations': 500, 'learning_rate': 0.03, 'depth': 4}
✅ RMSE: 0.0535, Score: 94.65026840
📁 Submission saved: Test_submission_F1_Top_P1_depth4_010500.csv

🚀 Running F1_Top_P2_depth6 with features: ['irradiance', 'soiling_ratio', 'current', 'power_output'] and params: {'iterations': 500, 'learning_rate': 0.05, 'depth': 6}
✅ RMSE: 0.0534, Score: 94.65865378
📁 Submission saved: Test_submission_F1_Top_P2_depth6_010502.csv

🚀 Running F1_Top_P3_depth5 with features: ['irradiance', 'soiling_ratio', 'current', 'power_output'] and params: {'iterations': 300, 'learning_rate': 0.07, 'depth': 5}
✅ RMSE: 0.0535, Score: 94.64743210
📁 Submission saved: Test_submission_F1_Top_P3_depth5_010503.csv

🚀 Running F2_Moderate_P1_depth4 with features: ['irradiance', 'soiling_ratio', 'current', 'power_output', 'panel_age', 'voltage', 'irradiance_per_cloud', 'temperature'] and params: {'iterations':

In [21]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from datetime import datetime
import numpy as np
import pandas as pd

# Define feature sets
features_1 = [  # Iteration 1
    "irradiance",
    "soiling_ratio",
    "current",
    "power_output"
]

features_2 = [  # Iteration 2
    "irradiance",
    "soiling_ratio",
    "current",
    "power_output",
    "panel_age",
    "voltage",
    "irradiance_per_cloud",
    "temperature"
]

features_3 = [  # Iteration 3
    "irradiance",
    "soiling_ratio",
    "current",
    "power_output",
    "panel_age",
    "voltage",
    "irradiance_per_cloud",
    "humidity",
    "module_temperature",
    "temperature"
]

feature_sets = {
    "Iteration1_Top": features_1,
    "Iteration2_Moderate": features_2,
    "Iteration3_Broader": features_3
}

# Run model training + prediction + submission for each feature set
for name, selected_features in feature_sets.items():
    print(f"\n🔹 Running {name} with features: {selected_features}")

    # Prepare data
    X = train[selected_features]
    y = train['efficiency']
    X_test = test[selected_features]

    # Split train/val
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train model
    model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=6, random_seed=42, verbose=False)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

    # Evaluate
    y_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    score = (1 - rmse) * 100
    print(f"✅ RMSE: {rmse:.4f}, Score: {score:.8f}")

    # Final prediction on test
    preds = model.predict(X_test)

    # Get current timestamp
    timestamp = datetime.now().strftime("%H%M%S")

    # Save submission
    submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})
    filename = f'Test_submission_{name}_{timestamp}.csv'
    submission.to_csv(filename, index=False)
    print(f"📁 Submission file saved as: {filename}")



🔹 Running Iteration1_Top with features: ['irradiance', 'soiling_ratio', 'current', 'power_output']
✅ RMSE: 0.0533, Score: 94.67131148
📁 Submission file saved as: Test_submission_Iteration1_Top_003811.csv

🔹 Running Iteration2_Moderate with features: ['irradiance', 'soiling_ratio', 'current', 'power_output', 'panel_age', 'voltage', 'irradiance_per_cloud', 'temperature']
✅ RMSE: 0.0461, Score: 95.39309910
📁 Submission file saved as: Test_submission_Iteration2_Moderate_003814.csv

🔹 Running Iteration3_Broader with features: ['irradiance', 'soiling_ratio', 'current', 'power_output', 'panel_age', 'voltage', 'irradiance_per_cloud', 'humidity', 'module_temperature', 'temperature']
✅ RMSE: 0.0449, Score: 95.50867066
📁 Submission file saved as: Test_submission_Iteration3_Broader_003817.csv
